In [28]:
from pathlib import Path
import re
import numpy as np
import glob
import os
import pandas as pd

BASE_DIR = Path.cwd()

if BASE_DIR.name == "scripts":
    BASE_DIR = BASE_DIR.parent
DATA_DIR = BASE_DIR / "data"

print("Current working directory:", Path.cwd())
print("BASE_DIR:", BASE_DIR)
print("DATA_DIR:", DATA_DIR)
print("DATA exists?", DATA_DIR.exists())

OUTPUT_DIR = BASE_DIR / "outputs"
OUTPUT_DIR.mkdir(exist_ok=True)


Current working directory: /home/chaopear/dev/r16-eeg-connectivity-analysis/scripts
BASE_DIR: /home/chaopear/dev/r16-eeg-connectivity-analysis
DATA_DIR: /home/chaopear/dev/r16-eeg-connectivity-analysis/data
DATA exists? True


In [29]:
base_path = Path(DATA_DIR).expanduser()
search_pattern = "*/Coherence/Complex Demodulation/*/*.conn"
conn_files = list(base_path.glob(search_pattern))

print("Number of .conn files found:", len(conn_files))

for file_path in conn_files[:5]:
    print(file_path)

Number of .conn files found: 448
/home/chaopear/dev/r16-eeg-connectivity-analysis/data/mandarin_adult_mms_conn/Coherence/Complex Demodulation/gu2/mms08m24_250Hz_gu2_AEP.conn
/home/chaopear/dev/r16-eeg-connectivity-analysis/data/mandarin_adult_mms_conn/Coherence/Complex Demodulation/gu2/mms14f25_250Hz_gu2_AEP.conn
/home/chaopear/dev/r16-eeg-connectivity-analysis/data/mandarin_adult_mms_conn/Coherence/Complex Demodulation/gu2/mms05f36_250Hz_gu2_AEP.conn
/home/chaopear/dev/r16-eeg-connectivity-analysis/data/mandarin_adult_mms_conn/Coherence/Complex Demodulation/gu2/mms09f24_250Hz_gu2_AEP.conn
/home/chaopear/dev/r16-eeg-connectivity-analysis/data/mandarin_adult_mms_conn/Coherence/Complex Demodulation/gu2/mms15m25_250Hz_gu2_AEP.conn


In [30]:
conn_path = conn_files[0]
print("Using file:", conn_path.name)

text = conn_path.read_text(encoding="utf-8", errors="ignore")

print(text[:500])


Using file: mms08m24_250Hz_gu2_AEP.conn
VersionNumber=1.0 DataType=Coherence DecompositionType=Complex Demodulation ConditionName=gu2 NumberTrials=156 NumberTimeSamples=31 TimeStartInMS=-500.0000 IntervalInMS=50.0000 NumberFrequencies=49 FreqStartInHz=2.0000 FreqIntervalInHz=1.0000 NumberChannels=15 
ACtL ACrL ACaL ACtR ACrR ACaR FL FR PL PR FpM FM CM PM OpM 
1.0000	1.0000	1.0000	1.0000	1.0000	1.0000	1.0000	1.0000	1.0000	1.0000	1.0000	1.0000	1.0000	1.0000	1.0000	1.0000	1.0000	1.0000	1.0000	1.0000	1.0000	1.0000	1.0000	1.0000	1.0000	1.0


In [59]:

def extract_participant_id(filename):
    """
    Extract the participant ID from the filename.
    For example:
    R16BF02_info1_250Hz_gu3_AEP.conn -> R16BF02
    M01y09_info1_250Hz_gu3_AEP.conn -> M01y09
    """
    return filename.split("_")[0].upper()


def extract_condition(filename):
    """
    Extract the participant ID from the filename.
    For example:
    R16BF02_info1_250Hz_gu3_AEP.conn -> gu3
    M01y09_info1_250Hz_gu3_AEP.conn -> gu3
    """
    return filename.split("_")[-2]


### Re-create meta file from conn file paths

In [69]:
def extract_group_info(directory_name):
    """
    Parses the directory name and returns a tuple of (language, age_group).
    """
    dir_lower = directory_name.lower()
    
    if 'eng' in dir_lower or 'ees' in dir_lower:
        language = 'English'
    elif 'man' in dir_lower or 'mms' in dir_lower:
        language = 'Mandarin'
    else:
        language = 'Unknown'
        
    if '5_7' in dir_lower:
        age_group = '5-7'
    elif '8_11' in dir_lower:
        age_group = '8-11'
    elif 'teen' in dir_lower:
        age_group = 'Teen'
    elif 'adult' in dir_lower:
        age_group = 'Adult'
    else:
        age_group = 'Unknown'
        
    return language, age_group

def read_meta_data_from_conn_file(conn_path):
    relative_path = conn_path.relative_to(base_path)
    parts = relative_path.parts
    
    # pattern is: <Group_Dir> / Coherence / Complex Demodulation / <Stimulus_Dir> / <Filename>
    # eg. ('2026-08-18_eng_children_5_7yrs_connect', 'Coherence', 'Complex Demodulation', 'gu1', 'R16C99b8.conn')
    group_dir = parts[0]
    stimulus_dir = parts[3]
    filename = parts[4]

    # print(f"Directory: {group_dir}")
    # print(f"Stimulus:  {stimulus_dir}")
    # print(f"Filename:  {filename}")
    # print("-" * 40)
    
    lang, age_group = extract_group_info(group_dir)
    
    row = {
        "participant_id": extract_participant_id(conn_path.name),
        "lang": lang,
        "age_group": age_group,
        "file_name": filename
    }

    return row

In [71]:
rows = []
failed_files = []

for conn_path in conn_files:
    try:
        row = read_meta_data_from_conn_file(conn_path)
        rows.append(row)
    except Exception as e:
        failed_files.append((conn_path.name, str(e)))

X_meta_all = pd.DataFrame(rows)

print("Successfully processed files:", len(rows))
print("Failed files:", len(failed_files))
print("Feature table shape:", X_meta_all.shape)


Successfully processed files: 448
Failed files: 0
Feature table shape: (448, 4)


In [72]:
X_meta = X_meta_all.drop_duplicates(
    subset=["participant_id"],
    keep="first"
).copy()

In [73]:

X_meta.to_csv(OUTPUT_DIR / "X_meta.csv", index=False)

print("\nMetadata prepared.")
print("metadata shape:", X_meta.shape)

print(X_meta.groupby('age_group').size())
print(X_meta.groupby('lang').size())


Metadata prepared.
metadata shape: (149, 4)
age_group
5-7      40
8-11     53
Adult    35
Teen     21
dtype: int64
lang
English     82
Mandarin    67
dtype: int64


### Extract conn data from conn files

In [4]:
def parse_conn_header(text):
    header_patterns = {
        "NumberTimeSamples": r"NumberTimeSamples=(\d+)",
        "NumberFrequencies": r"NumberFrequencies=(\d+)",
        "FreqStartInHz": r"FreqStartInHz=([0-9.]+)",
        "FreqIntervalInHz": r"FreqIntervalInHz=([0-9.]+)",
        "NumberChannels": r"NumberChannels=(\d+)",
    }

    header = {}

    for key, pattern in header_patterns.items():
        match = re.search(pattern, text)
        if match:
            value = match.group(1)
            header[key] = float(value) if "." in value else int(value)
        else:
            header[key] = None

    lines = text.splitlines()
    channels = None

    for i, line in enumerate(lines):
        if "NumberChannels=" in line:
            if i + 1 < len(lines):
                channels = lines[i + 1].strip().split()
            break

    header["channels"] = channels
    return header


header = parse_conn_header(text)

header

{'NumberTimeSamples': 31,
 'NumberFrequencies': 49,
 'FreqStartInHz': 2.0,
 'FreqIntervalInHz': 1.0,
 'NumberChannels': 15,
 'channels': ['ACtL',
  'ACrL',
  'ACaL',
  'ACtR',
  'ACrR',
  'ACaR',
  'FL',
  'FR',
  'PL',
  'PR',
  'FpM',
  'FM',
  'CM',
  'PM',
  'OpM']}

In [ ]:
def extract_numeric_data_after_channels(text):
    lines = text.splitlines()

    data_start_idx = None

    for i, line in enumerate(lines):
        if "NumberChannels=" in line:
            data_start_idx = i + 2
            break

    if data_start_idx is None:
        raise ValueError("Could not find NumberChannels line.")

    data_text = "\n".join(lines[data_start_idx:])

    numbers = re.findall(r"[-+]?\d*\.\d+|[-+]?\d+", data_text)
    values = np.array(numbers, dtype=float)

    return values


values = extract_numeric_data_after_channels(text)

expected_4d = (
    header["NumberTimeSamples"]
    * header["NumberFrequencies"]
    * header["NumberChannels"]
    * header["NumberChannels"]
)

expected_3d = (
    header["NumberFrequencies"]
    * header["NumberChannels"]
    * header["NumberChannels"]
)

print("Total numeric values:", len(values))
print("Expected 4D:", expected_4d)
print("Expected 3D:", expected_3d)
print("Match 4D?", len(values) == expected_4d)
print("Match 3D?", len(values) == expected_3d)

Total numeric values: 341775
Expected 4D: 341775
Expected 3D: 11025
Match 4D? True
Match 3D? False


In [6]:
n_time = header["NumberTimeSamples"]
n_freq = header["NumberFrequencies"]
n_ch = header["NumberChannels"]

if len(values) == expected_4d:
    data_4d = values.reshape(n_time, n_freq, n_ch, n_ch)
    print("Data shape:", data_4d.shape)

elif len(values) == expected_3d:
    data_4d = values.reshape(1, n_freq, n_ch, n_ch)
    print("Data had no time dimension. Converted shape:", data_4d.shape)

else:
    raise ValueError("Value count does not match expected 4D or 3D structure.")

Data shape: (31, 49, 15, 15)


In [7]:
data_freq_ch_ch = data_4d.mean(axis=0)

print("After averaging time:", data_freq_ch_ch.shape)

After averaging time: (49, 15, 15)


In [8]:
freqs = header["FreqStartInHz"] + np.arange(header["NumberFrequencies"]) * header["FreqIntervalInHz"]

bands = {
    "delta": (2, 4),
    "theta": (4, 8),
    "alpha": (8, 12),
    "beta": (12, 25),
    "highbeta": (25, 30),
    "gamma": (30, 45),
}

band_matrices = {}

for band_name, (low, high) in bands.items():
    idx = np.where((freqs >= low) & (freqs <= high))[0]
    band_matrices[band_name] = data_freq_ch_ch[idx].mean(axis=0)
    print(band_name, freqs[idx], band_matrices[band_name].shape)

delta [2. 3. 4.] (15, 15)
theta [4. 5. 6. 7. 8.] (15, 15)
alpha [ 8.  9. 10. 11. 12.] (15, 15)
beta [12. 13. 14. 15. 16. 17. 18. 19. 20. 21. 22. 23. 24. 25.] (15, 15)
highbeta [25. 26. 27. 28. 29. 30.] (15, 15)
gamma [30. 31. 32. 33. 34. 35. 36. 37. 38. 39. 40. 41. 42. 43. 44. 45.] (15, 15)


In [9]:
def extract_upper_triangle_features(band_matrices, channels):
    feature_dict = {}

    n_ch = len(channels)
    triu_idx = np.triu_indices(n_ch, k=1)

    for band_name, matrix in band_matrices.items():
        for i, j in zip(*triu_idx):
            ch1 = channels[i]
            ch2 = channels[j]
            feature_name = f"{band_name}_{ch1}_{ch2}"
            feature_dict[feature_name] = matrix[i, j]

    return feature_dict


features = extract_upper_triangle_features(band_matrices, header["channels"])

print("Number of features:", len(features))
list(features.items())[:10]

Number of features: 630


[('delta_ACtL_ACrL', np.float64(0.21234516129032263)),
 ('delta_ACtL_ACaL', np.float64(0.21640752688172038)),
 ('delta_ACtL_ACtR', np.float64(0.21915806451612907)),
 ('delta_ACtL_ACrR', np.float64(0.21723870967741932)),
 ('delta_ACtL_ACaR', np.float64(0.2126333333333333)),
 ('delta_ACtL_FL', np.float64(0.21127419354838714)),
 ('delta_ACtL_FR', np.float64(0.21266559139784946)),
 ('delta_ACtL_PL', np.float64(0.21088279569892476)),
 ('delta_ACtL_PR', np.float64(0.2112698924731183)),
 ('delta_ACtL_FpM', np.float64(0.21211720430107528))]

### Going throught all files:

In [ ]:

def read_conn_file_to_features(conn_path):
    text = conn_path.read_text(encoding="utf-8", errors="ignore")

    header = parse_conn_header(text)
    values = extract_numeric_data_after_channels(text)

    n_time = header["NumberTimeSamples"]
    n_freq = header["NumberFrequencies"]
    n_ch = header["NumberChannels"]

    expected_4d = n_time * n_freq * n_ch * n_ch
    expected_3d = n_freq * n_ch * n_ch

    if len(values) == expected_4d:
        data_4d = values.reshape(n_time, n_freq, n_ch, n_ch)
    elif len(values) == expected_3d:
        data_4d = values.reshape(1, n_freq, n_ch, n_ch)
    else:
        raise ValueError(
            f"{conn_path.name}: value count mismatch. "
            f"Got {len(values)}, expected {expected_4d} or {expected_3d}"
        )

    # Average time dimension
    data_freq_ch_ch = data_4d.mean(axis=0)

    # Frequency list
    freqs = header["FreqStartInHz"] + np.arange(header["NumberFrequencies"]) * header["FreqIntervalInHz"]

    bands = {
        "delta": (2, 4),
        "theta": (4, 8),
        "alpha": (8, 12),
        "beta": (12, 25),
        "highbeta": (25, 30),
        "gamma": (30, 45),
    }

    band_matrices = {}

    for band_name, (low, high) in bands.items():
        idx = np.where((freqs >= low) & (freqs <= high))[0]
        band_matrices[band_name] = data_freq_ch_ch[idx].mean(axis=0)

    features = extract_upper_triangle_features(band_matrices, header["channels"])

    row = {
        "participant_id": extract_participant_id(conn_path.name),
        "condition": extract_condition(conn_path.name),
        "filename": conn_path.name,
    }
    row.update(features)

    return row


rows = []
failed_files = []

for conn_path in conn_files:
    try:
        row = read_conn_file_to_features(conn_path)
        rows.append(row)
    except Exception as e:
        failed_files.append((conn_path.name, str(e)))

X_conn = pd.DataFrame(rows)

print("Successfully processed files:", len(rows))
print("Failed files:", len(failed_files))
print("Feature table shape:", X_conn.shape)

if failed_files:
    print("\nFailed file list:")
    for name, err in failed_files:
        print(name, "->", err)

X_conn.head()

Successfully processed files: 447
Failed files: 0
Feature table shape: (447, 633)


,participant_id,condition,filename,delta_ACtL_ACrL,delta_ACtL_ACaL,delta_ACtL_ACtR,delta_ACtL_ACrR,delta_ACtL_ACaR,delta_ACtL_FL,delta_ACtL_FR,...,gamma_FpM_FM,gamma_FpM_CM,gamma_FpM_PM,gamma_FpM_OpM,gamma_FM_CM,gamma_FM_PM,gamma_FM_OpM,gamma_CM_PM,gamma_CM_OpM,gamma_PM_OpM
0,EES15F31,gu2,ees15f31_250Hz_gu2_AEP.conn,0.212345,0.216408,0.219158,0.217239,0.212633,0.211274,0.212666,...,0.230584,0.231752,0.231410,0.229382,0.226628,0.224736,0.223512,0.229689,0.229848,0.220932
1,EES24F29,gu2,EES24F29_250Hz_gu2_AEP.conn,0.282070,0.282195,0.284954,0.281373,0.277839,0.280711,0.277217,...,0.295975,0.296160,0.298415,0.300288,0.289828,0.290241,0.290765,0.295141,0.298042,0.286973
2,EES12F24,gu2,ees12f24_250Hz_gu2_AEP.conn,0.250934,0.255659,0.251605,0.243583,0.244530,0.251760,0.246549,...,0.275361,0.273477,0.274041,0.277602,0.284012,0.285934,0.282308,0.278118,0.277761,0.283681
3,EES13M27,gu2,ees13m27_250Hz_gu2_AEP.conn,0.296286,0.298956,0.299748,0.304109,0.301956,0.300078,0.298660,...,0.277278,0.276235,0.275730,0.276121,0.276558,0.276355,0.277233,0.272575,0.272843,0.274815
4,EES21F20,gu2,EES21F20_250Hz_gu2_AEP.conn,0.255791,0.253618,0.250008,0.253818,0.252999,0.249192,0.247771,...,0.244407,0.245205,0.244573,0.243448,0.244233,0.246028,0.247893,0.244645,0.243673,0.243988


In [ ]:
x_output_path = OUTPUT_DIR / "X_conn_gu123.csv"

X_conn.to_csv(x_output_path, index=False)

print("Saved to:", x_output_path)

Saved to: /home/chaopear/dev/r16-eeg-connectivity-analysis/outputs/X_conn_gu123.csv


In [13]:
unique_pairs = X_conn.groupby(["participant_id", "condition"]).ngroups
print("Number of unique participants+cond:", unique_pairs)
print("Number of unique participants:", X_conn["participant_id"].nunique())
unique_ids_by_condition = X_conn.groupby("condition")["participant_id"].unique()
for condition, ids in unique_ids_by_condition.items():
    print(f"Condition {condition}:")
    print(f"Total Count: {len(ids)}\n")

X_conn[["participant_id", "filename"]].head(20)

Number of unique participants+cond: 447
Number of unique participants: 149
Condition gu1:
Total Count: 149

Condition gu2:
Total Count: 149

Condition gu3:
Total Count: 149



,participant_id,filename
0,EES15F31,ees15f31_250Hz_gu2_AEP.conn
1,EES24F29,EES24F29_250Hz_gu2_AEP.conn
2,EES12F24,ees12f24_250Hz_gu2_AEP.conn
3,EES13M27,ees13m27_250Hz_gu2_AEP.conn
4,EES21F20,EES21F20_250Hz_gu2_AEP.conn
5,EES22M27,EES22M27_250Hz_gu2_AEP.conn
6,EES10M22,EES10M22_250Hz_gu2_AEP.conn
7,EES14F26,ees14f26_250Hz_gu2_AEP.conn
8,EES17F38,EES17F38_250Hz_gu2_AEP.conn
9,EES20F22,ees20f22_250Hz_gu2_AEP.conn


### OBSOLETE: Process meta file for sample ids

In [14]:
META_PATH = BASE_DIR / "meta" / "R16 Participant information.xlsx"
meta_raw_minor = pd.read_excel(META_PATH, sheet_name='MostUpdated')
meta_raw_adult = pd.read_excel(META_PATH, sheet_name='Adults')

def make_canonical_id(pid):
    if pd.isna(pid):
        return np.nan

    pid = str(pid).strip()
    pid_upper = pid.upper()

    # R16A02g05 -> R16A02
    # R16C05y13speech -> R16C05
    # R16E08y05 -> R16E08
    match = re.match(r"^(R16[A-Z]+\d{2})", pid_upper)
    if match:
        return match.group(1)

    # R1629
    match = re.match(r"^(R16\d+)", pid_upper)
    if match:
        return match.group(1)

    return pid_upper


def make_age_group(age):
    if pd.isna(age):
        return np.nan
    elif 3 <= age < 8:
        return "3-7"
    elif 8 <= age < 13:
        return "8-12"
    elif 13 <= age < 20:
        return "Teens"
    else:
        return "Adults"

def extract_clean_meta(meta_raw):
    metadata_clean = meta_raw.copy()

    metadata_clean = metadata_clean.rename(columns={
        "Participant ID": "participant_id",
        "DOE/date of test": "date_of_test",
        "ID_ERP": "participant_id",
        "DOE": "date_of_test",
        "Lang": "lang",
        "AGE": "age"
    })

    metadata_clean = metadata_clean[
        [
            "participant_id",
            "lang",
            "DOB",
            "date_of_test"
        ]
    ].copy()

    metadata_clean["participant_id"] = metadata_clean["participant_id"].astype(str).str.strip().str.upper()
    metadata_clean["lang"] = metadata_clean["lang"].astype(str).str.strip()
    metadata_clean["lang"] = metadata_clean["lang"].replace(["nan", "NaN", "None", ""], np.nan)

    metadata_clean["DOB"] = pd.to_datetime(metadata_clean["DOB"], errors="coerce")
    metadata_clean["date_of_test"] = pd.to_datetime(metadata_clean["date_of_test"], errors="coerce")

    metadata_clean["age"] = (
        (metadata_clean["date_of_test"] - metadata_clean["DOB"]).dt.days / 365.25
    )

    metadata_clean["age_group"] = metadata_clean["age"].apply(make_age_group)

    metadata_clean["lang_binary"] = metadata_clean["lang"].apply(
        lambda x: "C" if x == "C" else ("Others" if pd.notna(x) else np.nan)
    )

    metadata_clean["exact_id"] = metadata_clean["participant_id"].astype(str).str.strip().str.upper()
    metadata_clean["canonical_id"] = metadata_clean["participant_id"].apply(make_canonical_id)
    metadata_clean["is_base_id"] = metadata_clean["exact_id"] == metadata_clean["canonical_id"]
    metadata_clean = metadata_clean.dropna(subset=['participant_id'])

    return metadata_clean

In [15]:
metadata_clean_minor =  extract_clean_meta(meta_raw_minor)
metadata_clean_adult =  extract_clean_meta(meta_raw_adult)

metadata_clean = pd.concat([metadata_clean_minor, metadata_clean_adult], ignore_index=True)

# Resolve duplicated canonical IDs by preferring base IDs
metadata_sorted = metadata_clean.sort_values(
    by=["canonical_id", "is_base_id"],
    ascending=[True, False]
)

metadata_unique = metadata_sorted.drop_duplicates(
    subset=["canonical_id"],
    keep="first"
).copy()



In [16]:
print(metadata_sorted.groupby('age_group').size())
print(metadata_sorted.groupby('lang').size())


age_group
3-7       48
8-12      55
Adults    44
Teens     14
dtype: int64
lang
A    12
C    74
E    46
S    32
dtype: int64


In [17]:
meta_ids = metadata_unique["participant_id"]
meta_ids

109      CSD499AB04
110      CSD499AH10
115       CSD499BR8
113      CSD499CS02
111      CSD499DV04
           ...     
103       R16S22B10
104        R16S23G8
91         R16S25B8
83           R16U01
85     UNKNOWN ID #
Name: participant_id, Length: 160, dtype: str

In [18]:
metadata_clean.to_csv(OUTPUT_DIR / "metadata_clean_all_conditions.csv", index=False)
metadata_unique.to_csv(OUTPUT_DIR / "metadata_unique_all_conditions.csv", index=False)

print("\nMetadata prepared.")
print("Clean metadata shape:", metadata_clean.shape)
print("Unique metadata shape:", metadata_unique.shape)


Metadata prepared.
Clean metadata shape: (164, 10)
Unique metadata shape: (160, 10)


In [19]:
meta_ids = metadata_clean["participant_id"]
conn_ids = X_conn["participant_id"]


In [20]:
sorted(set(meta_ids)-set(conn_ids))


['CSD499AB04',
 'CSD499AH10',
 'CSD499BR8',
 'CSD499CS02',
 'CSD499DV04',
 'CSD499E09',
 'CSD499K07',
 'CSD499KK01',
 'CSD499LS06',
 'CSD499SL03',
 'EES09M30MUSICIAN',
 'EES18M23',
 'EES20F22(WAS EEL09F21)',
 'EES23F23',
 'M01Y09',
 'MMS07F25',
 'MMS12M23*HEARING 6K-8KHZ',
 'MMS18M35',
 'NIHC01',
 'NIHC02',
 'R16A01Y08',
 'R16A02',
 'R16A03GO7',
 'R16A04',
 'R16A05ER3',
 'R16A06Y10',
 'R16A07',
 'R16BF01',
 'R16BF02',
 'R16BF04',
 'R16BF05',
 'R16BF06',
 'R16C03',
 'R16C04',
 'R16C05',
 'R16C06',
 'R16C06Y16',
 'R16C07???',
 'R16C08',
 'R16C09',
 'R16C10',
 'R16C11',
 'R16C12',
 'R16C13',
 'R16C15',
 'R16C16',
 'R16C17',
 'R16C20',
 'R16C22',
 'R16C23',
 'R16C24',
 'R16C25',
 'R16C26',
 'R16C27',
 'R16C28',
 'R16C29',
 'R16C30',
 'R16C31',
 'R16C32',
 'R16C33',
 'R16C34',
 'R16C35',
 'R16C36',
 'R16C37',
 'R16C38',
 'R16C39',
 'R16CJTEST',
 'R16CJY',
 'R16CRD',
 'R16CTEEN05',
 'R16CTEEN06',
 'R16E01',
 'R16E02',
 'R16E03 (ERP RECORD NAME R16R07)',
 'R16E04',
 'R16E05',
 'R16E06',
 'R16

In [21]:
print(sorted(set(conn_ids)-set(meta_ids)))

['CSD4990AH10F21', 'CSD499AB04G21', 'CSD499B8F21', 'CSD499CS02F20', 'CSD499DV04F21', 'CSD499E09F21', 'CSD499K07F21', 'CSD499KK01F21', 'CSD499LS06F20', 'CSD499SL03F20', 'EES09M30MU', 'EES20F22', 'EES23F24', 'M01Y09B9C', 'MMS07F25A', 'MMS12M23', 'MMS18M36MIX', 'NIH01C02B16', 'R16A01B08', 'R16A02G05', 'R16A03G07', 'R16A04B08', 'R16A05B08', 'R16A07B11', 'R16BF01B9', 'R16BF02G8', 'R16BF04G8', 'R16BF05B5', 'R16BF06G8', 'R16C03G6', 'R16C04B13', 'R16C05B13', 'R16C06B10', 'R16C06B16', 'R16C08G9', 'R16C09B8', 'R16C11B9', 'R16C12G7', 'R16C13B5', 'R16C15B7', 'R16C16B7', 'R16C17B6', 'R16C20B9', 'R16C22G10', 'R16C23B5', 'R16C24B7', 'R16C25B5', 'R16C26G10', 'R16C27B9', 'R16C28B6', 'R16C30B7', 'R16C31B7', 'R16C32B8', 'R16C33B5', 'R16C34B6', 'R16C35B6', 'R16C36B6', 'R16C37B6', 'R16C38G6', 'R16C39B10', 'R16CJTESTB15', 'R16CJY24B14', 'R16CM04G10', 'R16CM05', 'R16CM06', 'R16CM09', 'R16CP02XINGTG16', 'R16CTEEN06G16', 'R16E01G10', 'R16E02B6', 'R16E04G10', 'R16E05G10', 'R16E06G8', 'R16E07B9', 'R16E08B5', 'R1

In [22]:
metadata_clean

,participant_id,lang,DOB,date_of_test,age,age_group,lang_binary,exact_id,canonical_id,is_base_id
0,R16A01Y08,A,2014-04-07,2023-03-05,8.908966,8-12,Others,R16A01Y08,R16A01,False
1,R16A02,A,2018-10-14,2024-09-18,5.930185,3-7,Others,R16A02,R16A02,True
2,R16A03GO7,A,2017-04-30,2024-10-26,7.490760,3-7,Others,R16A03GO7,R16A03,False
3,R16A04,A,2015-09-23,2024-11-10,9.133470,8-12,Others,R16A04,R16A04,True
4,R16A05ER3,A,2016-07-19,2025-02-07,8.555784,8-12,Others,R16A05ER3,R16A05,False
...,...,...,...,...,...,...,...,...,...,...
159,MMS16M23,C,1988-06-07,2011-09-28,23.307324,Adults,C,MMS16M23,MMS16M23,True
160,MMS17M25,C,1985-10-15,2011-09-28,25.952088,Adults,C,MMS17M25,MMS17M25,True
161,MMS18M35,C,1976-10-26,2012-01-04,35.189596,Adults,C,MMS18M35,MMS18M35,True
162,MMS19F30,C,1982-04-20,2012-06-09,30.138261,Adults,C,MMS19F30,MMS19F30,True
